In [1]:
from pamda import pamda
import statistics
from pprint import pp

In [2]:
data = pamda.read_csv('outputs/geo_time_comparison_tests.csv')

In [3]:
columns = list(data[0].keys())
for i in columns:
    print(f"Column: {i}")

Column: city1
Column: city2
Column: coord1
Column: coord2
Column: test_world_highways_scgraph_time_ms
Column: test_world_highways_scgraph_length_km
Column: test_us_freeway_scgraph_time_ms
Column: test_us_freeway_scgraph_length_km
Column: test_haversine_time_ms
Column: test_haversine_length_km
Column: test_haversine_circuity_time_ms
Column: test_haversine_circuity_length_km
Column: test_google_time_ms
Column: test_google_length_km


In [4]:
time_columns = [i for i in columns if i.endswith('_time_ms')]
length_columns = [i for i in columns if i.endswith('_length_km')]

In [5]:
avg_times = {i: sum(item[i] for item in data) / len(data) for i in time_columns}
std_times = {i: statistics.stdev(item[i] for item in data) for i in time_columns if len(data) > 1}

print("Average Times:")
pp(avg_times)
print("Standard Deviation of Times:")
pp(std_times)

Average Times:
{'test_world_highways_scgraph_time_ms': 8.522971471150717,
 'test_us_freeway_scgraph_time_ms': 1.6002376874287922,
 'test_haversine_time_ms': 0.0028053919474283853,
 'test_haversine_circuity_time_ms': 0.0015676021575927734,
 'test_google_time_ms': 387.6781702041626}
Standard Deviation of Times:
{'test_world_highways_scgraph_time_ms': 4.389742988438339,
 'test_us_freeway_scgraph_time_ms': 0.793551040607762,
 'test_haversine_time_ms': 0.0015084404851258273,
 'test_haversine_circuity_time_ms': 0.0018853489832634316,
 'test_google_time_ms': 164.50920509837874}


In [6]:
distance_errors = {i: [item[i] - item['test_google_length_km'] for item in data] for i in length_columns}
avg_distance_errors = {i: sum(distance_errors[i]) / len(distance_errors[i]) for i in length_columns}
print("Average Distance Errors:")
pp(avg_distance_errors)
avg_length = {i: sum(item[i] for item in data) / len(data) for i in length_columns}
print("Average Route Lengths:")
pp(avg_length)
avg_pct_error = {i: avg_distance_errors[i] / avg_length[i] * 100 for i in length_columns}
print("Average Percentage Error:")
pp(avg_pct_error)

Average Distance Errors:
{'test_world_highways_scgraph_length_km': -103.79145623372203,
 'test_us_freeway_scgraph_length_km': 72.79806010537189,
 'test_haversine_length_km': -350.6028225081187,
 'test_haversine_circuity_length_km': 66.46328132359082,
 'test_google_length_km': 0.0}
Average Route Lengths:
{'test_world_highways_scgraph_length_km': 2332.1418854329445,
 'test_us_freeway_scgraph_length_km': 2508.7314017720387,
 'test_haversine_length_km': 2085.3305191585478,
 'test_haversine_circuity_length_km': 2502.3966229902576,
 'test_google_length_km': 2435.9333416666664}
Average Percentage Error:
{'test_world_highways_scgraph_length_km': -4.450477772472833,
 'test_us_freeway_scgraph_length_km': 2.901787734388428,
 'test_haversine_length_km': -16.81281788603902,
 'test_haversine_circuity_length_km': 2.655985094967481,
 'test_google_length_km': 0.0}


In [7]:
mape = {i: [abs(item[i] - item['test_google_length_km']) / item['test_google_length_km'] for item in data] for i in length_columns}
print("Mean Absolute Percentage Error (MAPE):")
pp({k:sum(v)/len(v) for k, v in mape.items()})

Mean Absolute Percentage Error (MAPE):
{'test_world_highways_scgraph_length_km': 0.041977860670147096,
 'test_us_freeway_scgraph_length_km': 0.042174538977986877,
 'test_haversine_length_km': 0.14368051949021463,
 'test_haversine_circuity_length_km': 0.04351754592628798,
 'test_google_length_km': 0.0}


In [8]:
def calculate_r_squared(y_true, y_pred):
    """
    Calculates the R-squared (coefficient of determination).

    Args:
        y_true (list): The true target values.
        y_pred (list): The predicted target values from the model.

    Returns:
        float: The R-squared value.
    """
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have the same length.")

    # Calculate the mean of the true values
    y_true_mean = sum(y_true) / len(y_true)

    # Calculate the total sum of squares (SST)
    sst = sum([(y - y_true_mean)**2 for y in y_true])

    # Calculate the residual sum of squares (SSR)
    ssr = sum([(y_true[i] - y_pred[i])**2 for i in range(len(y_true))])

    # Calculate R-squared
    if sst == 0:
        # Handle the case where there is no variance in y_true
        return 1.0 if ssr == 0 else float('-inf')
    else:
        return 1 - (ssr / sst)

In [9]:
r_sq = {i: calculate_r_squared([item['test_google_length_km'] for item in data], [item[i] for item in data]) for i in length_columns}
pp(r_sq)

{'test_world_highways_scgraph_length_km': 0.9899276200795405,
 'test_us_freeway_scgraph_length_km': 0.9898106625229094,
 'test_haversine_length_km': 0.8971260806296754,
 'test_haversine_circuity_length_km': 0.9912372731538726,
 'test_google_length_km': 1.0}
